In [ ]:
# This is required to run multiple processes on Unity for some reason.
from multiprocessing import set_start_method
set_start_method('spawn', force=True)

import os
os.environ['XLA_FLAGS'] = '--xla_gpu_enable_command_buffer='
os.environ['XLA_PYTHON_CLIENT_MEM_FRACTION'] = '0.5'
# os.environ['JAX_PLATFORMS'] = 'cpu'

In [1]:
%matplotlib widget
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import jax
import jax.numpy as jnp
from tqdm.auto import tqdm
from pathlib import Path
from importlib import reload

src = str(Path('../src').resolve())
if src not in sys.path:
    sys.path.append(src)
import config, data, models, train, evaluate

In [2]:
# reload(config)
from config import read_config

cfg, cfg_str = read_config("/work/pi_kandread_umass_edu/tss-ml/runs/caravan/base.yml")
cfg['num_workers'] = 2
cfg['log'] = False
cfg['quiet'] = False
cfg['use_cache'] = True
# cfg['batch_size'] = 256


In [3]:
dataset = data.HydroDataset(cfg)
# cfg = config.set_model_data_args(cfg, dataset)
# dataloader = data.HydroDataLoader(cfg, dataset)
# trainer = train.Trainer(cfg, dataloader=dataloader)
# trainer.start_training()

Loading static attributes
Loading dynamic data
Data Hash: 0bf72186d3d0de86f51017bc306a91640806e103abcf88fe088641d6e28c898c
Using cached basin dataset.


Updating Indices:   0%|          | 0/15960 [00:00<?, ?it/s]

In [ ]:
dataset.x_d

In [ ]:
plt.close('all')
dataset.x_d.sel(basin='camelsaus_401210')['streamflow'].plot()

In [ ]:
# Flexible data directory handling
data_dir = Path(cfg["data_dir"])
sub_dirs = cfg.get("data_sub_dirs")
if sub_dirs:
    data_dirs = [data_dir / d for d in sub_dirs]
else:
    data_dirs = [data_dir]

In [ ]:
data_dirs

In [ ]:
def read_file(fp):
    with open(fp, "r") as file:
        basin_list = file.readlines()
        basin_list = [basin.strip() for basin in basin_list]
        return basin_list
all_basins = read_file(cfg['data_dir'] / cfg['basin_file'])

In [ ]:
ts_dir_stem = cfg.get("time_series_dir", "time_series")
files = []
for data_dir in data_dirs:
    sub_dir_files = list((data_dir / ts_dir_stem).glob("*.nc"))[0:10]
    filt_files = [f for f in sub_dir_files if f.stem in all_basins]
    files.extend(filt_files)

In [ ]:
files

In [ ]:
data_dir = Path('/nas/cee-water/cjgleason/data/Caravan/output/Caravan-Jan25-nc/camelsaus')
all_files = list((data_dir/'time_series').glob("*.nc"))
# len(list(all_files))

In [ ]:
dataset.data_dirs

In [ ]:
filtered_files = [f for f in all_files if f.stem in dataset.all_basins]
filtered_files

In [ ]:
import xarray as xr
from tqdm.auto import tqdm

features = dataset.features['dynamic']['era5'][0:5]
files = []
# ds_list = []
for data_dir in dataset.data_dirs:
    files.extend(list((data_dir/'time_series').glob("*.nc"))[0:10])
    
def _preprocess(x):
    return x.sel(date=dataset.cfg["time_slice"])[features]

ds = xr.open_mfdataset(files, combine='nested', concat_dim='basin', preprocess=_preprocess)
    
ds['basin'] = [f.stem for f in files]

In [ ]:
ds

In [ ]:
ds = xr.open_mfdataset(files, combine='nested', concat_dim='basin')
ds

In [ ]:
ds['basin'].to_dataframe()['basin']

In [ ]:
dataset._normalize_data(ds, "dynamic", None, None)

In [ ]:
ds = xr.concat(ds_list, dim='basin')
ds

In [ ]:
ds.date

In [ ]:
xr.open_dataset(filtered_files[0])

In [ ]:
len(all_files)

In [ ]:
dataset.all_basins

In [ ]:
dataloader = data.HydroDataLoader(cfg, dataset)
for basin, date, batch in dataloader:
    break
batch

In [ ]:
(~dataset.x_d['dewpoint_temperature_2m_mean'].isnull()).mean(axis=1)

In [ ]:
(~dataset.x_d['snowmelt_sum'].isnull()).mean()

In [ ]:
(~dataset.x_d['Red'].isnull()).mean()

In [ ]:
for _, _, batch in dataloader:
    break
key = jax.random.PRNGKey(0)

In [ ]:
keys = jax.random.split(key, cfg['batch_size'])
y_pred = jax.vmap(trainer.model)(batch,keys)

y_pred.shape

In [ ]:
x_data = batch['dynamic']['era5'][0,...]
s_data = batch['static'][0,...]

In [ ]:
dataset.seq2seq

In [ ]:
trainer.cfg['model_args']

In [ ]:
model = trainer.model
x = jax.vmap(model)(batch, keys)


In [ ]:
batch['y'].shape

In [ ]:
dataset.seq2seq

In [ ]:
df = dataset.x_d[['ssc','flux','usgs_q']].to_dataframe()
sites = df.groupby('basin').count()

In [ ]:
x_d = batch['dynamic']['era5'][0,...]
x_s = batch['static'][0,...]

x_s_broadcast = jnp.broadcast_to(x_s, (x_d.shape[0], x_s.shape[0]))
tmp = jnp.concatenate([x_d, x_s_broadcast], axis=-1)

In [ ]:
tmp.shape

In [ ]:
plt.close('all')
plt.imshow(tmp)

In [ ]:
for _, _, batch in dataloader:
    break
    
batch

In [ ]:
(df.groupby('basin').count()>0).sum(axis=1).value_counts()

In [ ]:
df.xs('USGS-14312260', level='basin').dropna(how='all')

In [ ]:
sites[(sites['flux']>0) & (sites['ssc']>0) & (sites['usgs_q']==0)]

In [ ]:

sites[(sites['flux']==0) & (sites['usgs_q']==0)]

In [ ]:
reload(train)




In [ ]:
reload(train)
from train import Trainer

log_dir = Path("/work/pi_kandread_umass_edu/tss-ml/runs/ssf_smac_opt2/very_quick_test_20250402_183045")
trainer = Trainer.load_last_checkpoint(log_dir)

In [ ]:
trainer.model

In [ ]:
trainer.early_stopper(2)

In [ ]:
ll = trainer.early_stopper.loss_list
best = ll[0]
ri = []
for l in ll:
    imp = (best - l) / abs(best)
    best = l if imp > 0.01 else best
    ri.append(imp > 0.01)
ri

In [ ]:
plt.close('all')
plt.plot(trainer.early_stopper.loss_list)

In [ ]:
trainer.early_stopper.threshold * trainer.early_stopper.best_loss

In [ ]:
for basin, date, batch in dataloader:
    break
batch

In [ ]:
batch['dynamic']['era5']

In [ ]:
trainer.v_losses

In [ ]:
losses = [2.9751103, 2.6642797, 2.7429867, 2.7114182, 2.6989262, 2.6787248]
losses = np.array(losses)
losses

In [ ]:
losses[-5:-3]

In [ ]:
for i in range(1, 5+2): #Changed to +2, so that the current loss is included.
    temp_best_loss = min(losses[:-i])
    current_loss = self.v_losses[-i]

    # Edge case to avoid division by 0.
    if temp_best_loss == 0:
        return True 

    relative_improvement = (temp_best_loss - current_loss) / abs(temp_best_loss)
    if relative_improvement >= criteria['threshold']:
        return False

In [ ]:
min_recent_loss

In [ ]:
current_loss

In [ ]:
losses[-5:]

In [ ]:
trainer.model

In [ ]:
cfg, model, trainer_state, opt_state, _  = train.load_last_state(Path('/work/pi_kandread_umass_edu/tss-ml/runs/Flex_TEALSTM/all_data_20241105_181617'))
cfg['log'] = False
cfg['exclude_target_from_index'] = None
cfg['quiet'] = False
dataset = HydroDataset(cfg)
dataset.update_indices('test')
dataloader = HydroDataLoader(cfg, dataset)

results = evaluate.predict(model, dataloader, return_dt=True, quiet=False, denormalize=True)
len(results)

In [ ]:
results

In [ ]:
results.index.get_level_values('basin').unique()

In [ ]:
results

In [ ]:
key = jax.random.PRNGKey(0)
batch_keys = jax.random.split(key, cfg['batch_size'])

dataloader = HydroDataLoader(cfg, dataset)
for basin, date, batch in tqdm(dataloader):
    batch = trainer.dataloader.shard_batch(batch)
    
    loss, grads, new_model, new_opt_state = train.step.make_step(
        trainer.model, 
        batch,
        batch_keys,
        trainer.opt_state, 
        trainer.optim, 
        trainer.filter_spec, 
        trainer.dataloader.dataset.denormalize_target,
        **trainer.cfg['step_kwargs']
    )
    
    if np.isnan(loss):
        break
    else:
        trainer.model = new_model
        trainer.opt_state = new_opt_state

In [ ]:
np.isnan(loss)

In [ ]:
batch = trainer.dataloader.shard_batch(batch)
batch['dynamic_dt']['era5'].device

In [ ]:
cell = trainer.model.encoders['landsat'].cell
cell

In [ ]:
cfg['model_args']['

In [ ]:
x_d = batch['dynamic']['landsat'][0]
x_s = batch['static'][0]
static_bias = trainer.model.static_embedder(x_s, key)

h_0 = jnp.zeros(cfg['model_args']['hidden_size'])
dt = 1

time_weight = cell.time_distance(dt, static_bias)
time_weight.shape

In [ ]:
# lax.scan 
for x in x_d:
    if jnp.all(~jnp.isnan(x)):
        break
x

In [ ]:
gates = jnp.dot(x, cell.weight_ih.T) + jnp.dot(h_0, cell.weight_hh.T) + cell.bias
f, g, o = jnp.split(gates, 3, axis=-1)
i = jax.nn.sigmoid(cell.input_linear(static_bias))

f = jax.nn.sigmoid(f) * time_weight

In [ ]:
import jax.numpy as jnp

params = jnp.split(trainer.model.encoders['landsat'].cell.decay_weights, 2, axis=0)

In [ ]:
params[0,:].shape

In [ ]:
trainer.model.encoders['landsat'].cell

In [ ]:
batch = dataloader.shard_batch(batch)

batch['dynamic_dt']['landsat'].device

In [ ]:
first_values.shape

In [ ]:
last_valid_index.shape

In [ ]:
x_d = batch['dynamic']['landsat']
# x_s = batch['static'][0]



valid_mask = np.all(~np.isnan(x_d),axis=2)

# valid_mask[0] = True

indices = np.arange(valid_mask.shape[1])
                  
valid_indices = np.where(valid_mask, indices, -1)


last_valid_index = np.maximum.accumulate(valid_indices, axis=1) 

first_values = valid_mask[:,0].astype(int)[:, None]
dt = np.concat([first_values, np.diff(last_valid_index, axis=1)],axis=1)


dt[1]


In [ ]:
dt[0]

In [ ]:
plt.close('all')
plt.imshow(dt)#,aspect='auto')

In [ ]:
alt_cfg = cfg.copy()
alt_cfg['batch_size'] = 1
dataloader = TAPDataLoader(alt_cfg, dataset)

for basin, date, batch in dataloader:
    break
    
single_data = {k: v[0,...] for k, v in batch.items()}

In [ ]:
trainer.model(single_data, jax.random.PRNGKey(0))

In [ ]:
batch['y'][0,...].shape

In [ ]:
# Resume training. Either directly from memory or loading a checkpoint.
import optax

# trainer.load_state('epoch100')
# trainer.load_last_state()

more_epochs = 0
new_schedule = optax.exponential_decay(0.01, trainer.epoch+more_epochs, 0.001, transition_begin=trainer.epoch)
trainer.lr_schedule = new_schedule
trainer.num_epochs += more_epochs

#Have to make a new dataloader when the last one is interrupted. 
trainer.dataloader = TAPDataLoader(cfg, dataset) 
trainer.start_training() 

In [ ]:
reload(evaluate)
from evaluate import predict, get_all_metrics

# basin = np.random.choice(dataset.basins).tolist()
basin = 'USGS-09367540'

cfg['data_subset'] = 'predict'
cfg['basin_subset'] =  basin
cfg['num_workers'] = 0 # Faster for small runs
dataloader = TAPDataLoader(cfg, dataset)

results = predict(trainer.model, dataloader, seed=0, denormalize=True)
results['pred'] = results['pred'] * (results['pred']>0) #Clip predictions to 0

results = results.reset_index()
results = results.sort_values(by='date')
results = results.drop(columns=['basin'], axis=1, level=0)
results.set_index('date', inplace=True)


In [ ]:
feature = 'usgs_q'

# Plot the true values and predictions
fig, ax = plt.subplots(figsize=(12, 6))
results['pred'][feature].plot(ax=ax)
results['obs'][feature].plot(ax=ax,linestyle='None',marker='.')

plt.title(f"Basin: {basin}")
plt.legend()
fig.autofmt_xdate()
# plt.ylim([0,20000])
plt.show()

In [ ]:
basin

In [ ]:
"""
'USGS-09367540'
"""

In [ ]:
results.plot.scatter('obs','pred')
plt.gca().axis('square')
# plt.xscale('log')
# plt.yscale('log')
# plt.xlim([0,20])
# plt.ylim([0,20])
plt.show()

In [ ]:
import train
from data import TAPDataset, TAPDataLoader

state_dir = Path("../runs/notebook/20240603_1359/epoch18")
cfg, model, trainer_state, opt_state = train.load_state(state_dir)
dataset = TAPDataset(cfg)

In [ ]:
reload(evaluate)
from evaluate import predict, get_all_metrics

cfg['data_subset'] = 'test'
cfg['num_workers'] = 4
dataloader = TAPDataLoader(cfg, dataset)

results = predict(model, dataloader, seed=0, denormalize=True)
results['pred'] = results['pred']# * (results['pred']>0) #Clip predictions to 0

# results = results.reset_index()
# results = results.sort_values(by='date')

metrics = get_all_metrics(results['obs'],results['pred'])
metrics

In [ ]:
%matplotlib widget
plt.close('all')
plt.scatter(batch['y'][...,-1],pred[...,-1])
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10,3))
xd = axes[0].imshow(batch['x_dd'][:,:,0],aspect='auto')
fig.colorbar(xd, ax=axes[0])
xs = axes[1].imshow(batch['x_s'],aspect='auto')
fig.colorbar(xs, ax=axes[1]) 

In [ ]:
batch['x_dd'][:,:,0].shape

In [ ]:
basins[idx_max_err]

In [ ]:
positional_encoding = trainer.model.d_encoder.embedder.positional_encoding

plt.figure(figsize=(10, 8))
plt.imshow(positional_encoding, cmap='viridis')
plt.xlabel('Embedding Dimension')
plt.ylabel('Position')
plt.title('Positional Encodings')
plt.show()